<a href="https://colab.research.google.com/github/hollymunck-glitch/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

## Business Scenario

You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.


Your task is to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Dataset Description: Hotel Bookings

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance

### Data Dictionary

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Load the `hotels.csv` file
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# Load the dataset
df = pd.read_csv('/content/hotels.csv')

# Display initial info
print("Initial shape:", df.shape)
print("\nMissing values before handling:")
print(df.isnull().sum())

# Handle missing values
# Impute 'children' with the mean
df['children'].fillna(df['children'].mean(), inplace=True)
# Impute 'country' with the mode
df['country'].fillna(df['country'].mode()[0], inplace=True)
# Impute 'agent' and 'company' with 0 as per data description
df['agent'].fillna(0, inplace=True)
df['company'].fillna(0, inplace=True)

# Remove rows with missing 'adr' as it's a small number of rows
df.dropna(subset=['adr'], inplace=True)

print("\nMissing values after handling:")
print(df.isnull().sum())
print("\nShape after handling missing values:", df.shape)

# Separate target variable
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object', 'category']).columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

# Create a preprocessing pipeline for encoding and keeping numerical features
# Explicitly define transformers for both categorical and numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numerical_features) # Use 'passthrough' for numerical features
    ],
    remainder='drop' # Drop any other columns not specified
)

# Fit and transform the data, converting to a dense array
X_processed = preprocessor.fit_transform(X).toarray()


# Get feature names after preprocessing
# This method should correctly get the names from both transformers
all_feature_names = preprocessor.get_feature_names_out()

# Convert the processed data back to a DataFrame with the correct column names
X_processed_df = pd.DataFrame(X_processed, columns=all_feature_names, index=X.index)


# Split data into training and test sets (70/30)
X_train, X_test, y_train, y_test = train_test_split(X_processed_df, y, test_size=0.3, random_state=42)

print("\nShape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Initial shape: (119390, 32)

Missing values before handling:
hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                         

/tmp/ipython-input-3347086153.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['children'].fillna(df['children'].mean(), inplace=True)
/tmp/ipython-input-3347086153.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tr

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
company                           0
days_in_waiting_list              0
customer_type                     0
adr                         

### ✍️ Your Response: 🔧
1. The dataset has 119,390 rows and 32 columns, and that didn’t change after cleaning.

2. It includes both categorical features (like hotel, arrival_date_month, country) and numerical ones (like lead_time, adults, children, adr).

3. To prepare the data, I loaded the file, filled in missing values (mean for children, mode for country, and 0 for agent and company), and dropped rows missing adr. Then I separated the target (is_canceled), encoded the categorical variables with One-Hot Encoding, kept the numerical ones as they were, and split the data into 70% training and 30% testing sets.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

### In Your Response:
1. How accurate is this model?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [6]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize and train the Naïve Bayes model
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict on the test data
y_pred_nb = nb_model.predict(X_test)

# Print classification report and confusion matrix
print("Naïve Bayes Model Evaluation:")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

# Store accuracy for comparison later
nb_accuracy = nb_model.score(X_test, y_test)
print(f"\nNaïve Bayes Accuracy: {nb_accuracy:.4f}")

Naïve Bayes Model Evaluation:

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     22478
           1       1.00      1.00      1.00     13339

    accuracy                           1.00     35817
   macro avg       1.00      1.00      1.00     35817
weighted avg       1.00      1.00      1.00     35817


Confusion Matrix:
[[22478     0]
 [    0 13339]]

Naïve Bayes Accuracy: 1.0000


### ✍️ Your Response: 🔧
1. The classification report shows an accuracy of 1.00 (100%) on the test data. The confusion matrix shows that the model correctly predicted all instances of both canceled (1) and not canceled (0) bookings.

2. If the model’s high accuracy is real, it could be very useful for the hotel. It could flag likely cancellations in real time, help with staffing and inventory decisions, support pricing adjustments for revenue management, and guide marketing toward guests less likely to cancel. However, perfect accuracy is unusual and might mean there’s data leakage or another issue, so it should be double-checked before being used in practice.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Train an SVM classifier (use RBF kernel)
- Make predictions and evaluate with classification metrics

### In Your Response:
1. How well does the model perform?
2. In what business situations could SVM provide better insights than simpler models?


In [7]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
# Due to the large dataset size and the complexity of SVM,
# we will use a subset of the data for demonstration purposes.
# In a real-world scenario, you might consider techniques like
# stochastic gradient descent SVM (SGDClassifier with loss='hinge')
# or distributed computing for large datasets.
subset_size = 10000
X_train_subset = X_train.sample(n=subset_size, random_state=42)
y_train_subset = y_train.sample(n=subset_size, random_state=42)
X_test_subset = X_test.sample(n=int(subset_size * 0.3), random_state=42)
y_test_subset = y_test.sample(n=int(subset_size * 0.3), random_state=42)


# Initialize and train the SVM model with RBF kernel
# Using a smaller C value and gamma='auto' for potentially faster convergence on a subset
svm_model = SVC(kernel='rbf', C=1, gamma='auto')
print("Training SVM model on a subset of the data...")
svm_model.fit(X_train_subset, y_train_subset)
print("SVM model training complete.")

# Predict on the test subset
y_pred_svm = svm_model.predict(X_test_subset)

# Print classification report and confusion matrix
print("\nSupport Vector Machine (SVM) Model Evaluation (Subset Data):")
print("\nClassification Report:")
print(classification_report(y_test_subset, y_pred_svm))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test_subset, y_pred_svm))

# Store accuracy for comparison later
svm_accuracy = svm_model.score(X_test_subset, y_test_subset)
print(f"\nSVM Accuracy (Subset Data): {svm_accuracy:.4f}")

Training SVM model on a subset of the data...
SVM model training complete.

Support Vector Machine (SVM) Model Evaluation (Subset Data):

Classification Report:
              precision    recall  f1-score   support

           0       0.76      0.92      0.83      1925
           1       0.77      0.48      0.59      1075

    accuracy                           0.76      3000
   macro avg       0.76      0.70      0.71      3000
weighted avg       0.76      0.76      0.74      3000


Confusion Matrix:
[[1767  158]
 [ 560  515]]

SVM Accuracy (Subset Data): 0.7607


### ✍️ Your Response: 🔧
1. The SVM model reached about 76% accuracy, with 0.77 precision and 0.48 recall for predicting cancellations. So, it’s good at correctly labeling cancellations when it predicts them, but it misses some actual ones. Its results are much lower than the Naïve Bayes model’s perfect accuracy, which likely needs to be rechecked for possible data issues.

2. SVMs can be useful when relationships between features are complex or non-linear, when the business wants a clear separation between likely and unlikely cancellations, or when working with many features after encoding. However, they’re slower on large datasets and harder to interpret than simpler models.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLBClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Evaluate accuracy and performance

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [8]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Neural Networks are sensitive to feature scaling, so we will scale the data
# We will use the full training and test sets here, but scaling is crucial.
# Fit the scaler on the training data and transform both training and test data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Initialize the MLPClassifier
# Using a simple architecture with two hidden layers of size 50 each
# Increased max_iter for better convergence
mlp_model = MLPClassifier(hidden_layer_sizes=(50, 50), max_iter=300, random_state=42, early_stopping=True)

print("Training Neural Network model...")
# Train the model on the scaled training data
mlp_model.fit(X_train_scaled, y_train)
print("Neural Network model training complete.")

# Predict on the scaled test data
y_pred_mlp = mlp_model.predict(X_test_scaled)

# Print classification report and confusion matrix
print("\nNeural Network Model Evaluation:")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_mlp))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))

# Store accuracy for comparison later
mlp_accuracy = mlp_model.score(X_test_scaled, y_test)
print(f"\nNeural Network Accuracy: {mlp_accuracy:.4f}")

Training Neural Network model...
Neural Network model training complete.

Neural Network Model Evaluation:

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     22478
           1       1.00      1.00      1.00     13339

    accuracy                           1.00     35817
   macro avg       1.00      1.00      1.00     35817
weighted avg       1.00      1.00      1.00     35817


Confusion Matrix:
[[22466    12]
 [    6 13333]]

Neural Network Accuracy: 0.9995


### ✍️ Your Response: 🔧
1. The Neural Network reached about 99.95% accuracy, which is higher than the SVM’s 76% and close to the Naïve Bayes model’s perfect score. It made very few mistakes in predicting both cancellations and non-cancellations, though the near-perfect results might need a closer look for possible data issues.

2. It depends on the business’s priorities. If the goal is accuracy and efficiency, they might use it even if they don’t fully understand how it works. But if explainability and trust are important, like wanting to know why guests cancel or needing to justify decisions, a simpler model would make more sense. The accuracy is impressive, but it should be verified before the model is relied on.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [9]:
# Print the stored accuracies
print(f"Naïve Bayes Accuracy: {nb_accuracy:.4f}")
print(f"SVM Accuracy (Subset Data): {svm_accuracy:.4f}")
print(f"Neural Network Accuracy: {mlp_accuracy:.4f}")

# Determine and print the best performing model based on accuracy
# Need to consider if comparing subset SVM to full data NB/NN is appropriate for a final conclusion
# For the purpose of this comparison based on the current results:
print("\nComparison Summary:")
if mlp_accuracy > nb_accuracy and mlp_accuracy > svm_accuracy:
    print("The Neural Network model had the highest accuracy.")
elif nb_accuracy > mlp_accuracy and nb_accuracy > svm_accuracy:
     print("The Naïve Bayes model had the highest accuracy.")
elif svm_accuracy > mlp_accuracy and svm_accuracy > nb_accuracy:
    print("The SVM model had the highest accuracy (on the subset data).")
else:
    print("Accuracy scores are very close or tied between models.")

Naïve Bayes Accuracy: 1.0000
SVM Accuracy (Subset Data): 0.7607
Neural Network Accuracy: 0.9995

Comparison Summary:
The Naïve Bayes model had the highest accuracy.


### ✍️ Your Response: 🔧
1. Naïve Bayes reached 100% accuracy, which is likely a sign of data issues and should be checked. The Neural Network followed closely at about 99.95%, and the SVM scored around 76%. While the Neural Network performed best overall, the near-perfect results from both it and Naïve Bayes seem suspicious and worth double-checking. Naïve Bayes trains the fastest and is easy to interpret, but the results might not be reliable. SVMs take longer to train and are harder to interpret, especially with large datasets. Neural Networks can take more setup and tuning but often perform better once optimized, though they’re basically a black box.

2. I wouldn’t recommend deploying any of the models just yet. The perfect accuracy from Naïve Bayes and the Neural Network probably points to data leakage or a preprocessing issue. The SVM’s lower performance also makes it less ideal. I’d first review how the data was cleaned and prepared to confirm the results are valid. If the data checks out, the Neural Network would likely be the best choice, but only if the business is okay with not fully understanding how the model makes its predictions.

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. While initial results show near-perfect accuracy for predicting cancellations with Neural Networks (and Naïve Bayes), this likely indicates a data issue needing investigation before deployment. If confirmed valid, a Neural Network could significantly reduce cancellation impact by enabling proactive measures. However, its "black box" nature limits understanding why cancellations occur. Future data on guest feedback or reason for cancellation could enhance insights and model performance, potentially favoring more interpretable models or explainable AI techniques.


2. This connects to my customized learning outcome because it shows the importance of critically analyzing data rather than just accepting results. Like in accounting, I’m learning to question accuracy, identify potential errors, and look deeper into what drives outcomes. This mindset helps me think analytically, apply data-driven reasoning, and make more reliable business decisions instead of relying only on surface level results.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_LastnameFirstname.ipynb"